In [1]:
"""
MPS and DMRG Implementation
Based on: U. Schollwöck, "The density-matrix renormalization group in the age of matrix product states"
Annals of Physics 326 (2011) 96-192

This implementation follows the LaTeX note specifications exactly.
"""

import numpy as np
from scipy.linalg import svd, qr
from scipy.sparse.linalg import LinearOperator, eigsh
import warnings
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# =============================================================================
# MPS Data Structures
# =============================================================================

class MPS:
    """Matrix Product State.

    Storage: sites[i][a_left, sigma, a_right]
    """
    def __init__(self, tensors):
        self.sites = [np.asarray(t, dtype=complex) for t in tensors]
        self.L = len(tensors)

    def copy(self):
        return MPS([s.copy() for s in self.sites])


def compute_energy(mps, mpo):
    """Compute <psi|H|psi> for MPS with respect to MPO."""
    D_W = mpo[0].shape[1] if mpo[0].shape[0] == 1 else mpo[0].shape[0]
    result = np.zeros((D_W, 1, 1), dtype=complex)
    result[-1, 0, 0] = 1.0

    for i in range(mps.L):
        result = np.einsum('bxy,xsa,bBst,ytA->BaA', result, mps.sites[i], 
                          mpo[i], mps.sites[i].conj())

    return float(result[0, 0, 0].real)


# =============================================================================
# Environment Updates (Equations 2-3 from LaTeX note)
# =============================================================================

def update_left_env(L_old, A, W):
    """Update left environment from site i-1 to i.

    L_new[B, a, A] = sum L_old[b, x, y] * A[x, s, a] * W[b, B, s, t] * A*[y, t, A]
    """
    return np.einsum('bxy,xsa,bBst,ytA->BaA', L_old, A, W, A.conj())


def update_right_env(R_old, B, W):
    """Update right environment from site i+1 to i.

    R_new[b, a, A] = sum B[a, s, x] * R_old[B, x, y] * W[b, B, s, t] * B*[A, t, y]
    """
    return np.einsum('asx,Bxy,bBst,Aty->baA', B, R_old, W, B.conj())


# =============================================================================
# MPO Construction
# =============================================================================

def heisenberg_mpo(L, J=1.0, Jz=1.0, h=0.0):
    """Construct Heisenberg MPO for spin-1/2 chain.

    H = sum_i [J/2(S+_i S-_{i+1} + S-_i S+_{i+1}) + Jz S^z_i S^z_{i+1} - h S^z_i]
    """
    d = 2
    D_W = 5

    Sp = np.array([[0., 1.], [0., 0.]], dtype=complex)
    Sm = np.array([[0., 0.], [1., 0.]], dtype=complex)
    Sz = np.array([[0.5, 0.], [0., -0.5]], dtype=complex)
    Id = np.eye(2, dtype=complex)

    mpo = []

    # Site 0 (left boundary)
    W = np.zeros((1, D_W, d, d), dtype=complex)
    W[0, 0] = -h * Sz
    W[0, 1] = (J/2) * Sm
    W[0, 2] = (J/2) * Sp
    W[0, 3] = Jz * Sz
    W[0, 4] = Id
    mpo.append(W)

    # Bulk sites
    for i in range(1, L-1):
        W = np.zeros((D_W, D_W, d, d), dtype=complex)
        W[0, 0] = Id
        W[1, 0] = Sp
        W[2, 0] = Sm
        W[3, 0] = Sz
        W[4, 0] = -h * Sz
        W[4, 1] = (J/2) * Sm
        W[4, 2] = (J/2) * Sp
        W[4, 3] = Jz * Sz
        W[4, 4] = Id
        mpo.append(W)

    # Site L-1 (right boundary)
    W = np.zeros((D_W, 1, d, d), dtype=complex)
    W[0, 0] = Id
    W[1, 0] = Sp
    W[2, 0] = Sm
    W[3, 0] = Sz
    W[4, 0] = -h * Sz
    mpo.append(W)

    return mpo


def aklt_mpo(L):
    """Construct AKLT MPO for spin-1 chain.

    H = sum_i [S_i . S_{i+1} + (1/3)(S_i . S_{i+1})^2]
    """
    d = 3
    D_W = 5

    # Spin-1 operators
    Sp = np.array([[0., np.sqrt(2), 0.],
                   [0., 0., np.sqrt(2)],
                   [0., 0., 0.]], dtype=complex)

    Sm = np.array([[0., 0., 0.],
                   [np.sqrt(2), 0., 0.],
                   [0., np.sqrt(2), 0.]], dtype=complex)

    Sz = np.array([[1., 0., 0.],
                   [0., 0., 0.],
                   [0., 0., -1.]], dtype=complex)

    Id = np.eye(3, dtype=complex)

    mpo = []

    # Site 0
    W = np.zeros((1, D_W, d, d), dtype=complex)
    W[0, 0] = (2.0/3.0) * Id
    W[0, 1] = Sm / np.sqrt(2)
    W[0, 2] = Sp / np.sqrt(2)
    W[0, 3] = Sz
    W[0, 4] = Id
    mpo.append(W)

    # Bulk
    for i in range(1, L-1):
        W = np.zeros((D_W, D_W, d, d), dtype=complex)
        W[0, 0] = Id
        W[1, 0] = Sp / np.sqrt(2)
        W[2, 0] = Sm / np.sqrt(2)
        W[3, 0] = Sz
        W[4, 0] = (2.0/3.0) * Id
        W[4, 1] = Sm / np.sqrt(2)
        W[4, 2] = Sp / np.sqrt(2)
        W[4, 3] = Sz
        W[4, 4] = Id
        mpo.append(W)

    # Site L-1
    W = np.zeros((D_W, 1, d, d), dtype=complex)
    W[0, 0] = Id
    W[1, 0] = Sp / np.sqrt(2)
    W[2, 0] = Sm / np.sqrt(2)
    W[3, 0] = Sz
    W[4, 0] = (2.0/3.0) * Id
    mpo.append(W)

    return mpo


# =============================================================================
# Exact Diagonalization (for small systems)
# =============================================================================

def exact_diagonalization(mpo):
    """Compute exact ground state energy by full diagonalization."""
    L = len(mpo)
    d = mpo[0].shape[2]
    D_W = 5

    H = np.zeros((d**L, d**L), dtype=complex)

    def add_term(site, bond_in, current):
        nonlocal H
        if site == L:
            if bond_in == 0:
                H += current
            return

        for bond_out in range(D_W):
            if site == 0:
                W = mpo[site][0, bond_out]
            elif site == L - 1:
                if bond_in >= mpo[site].shape[0]:
                    continue
                W = mpo[site][bond_in, 0]
            else:
                if bond_in >= mpo[site].shape[0]:
                    continue
                W = mpo[site][bond_in, bond_out]

            if np.all(np.abs(W) < 1e-14):
                continue

            add_term(site + 1, bond_out, np.kron(current, W))

    add_term(0, 0, np.array([[1.0]], dtype=complex))

    eigvals = np.linalg.eigvalsh(H.real)
    return eigvals[0]


# =============================================================================
# Single-Site DMRG (Matrix-Free with Lanczos)
# =============================================================================

class SingleSiteDMRG:
    """Single-site DMRG with matrix-free effective Hamiltonian."""

    def __init__(self, mpo, D_max=20):
        self.mpo = mpo
        self.L = len(mpo)
        self.d = mpo[0].shape[2]
        self.D_max = D_max
        self.D_W = 5

        # Initialize with random MPS
        tensors = []
        D = 1
        for i in range(self.L):
            D_next = min(D_max, D * self.d) if i < self.L - 1 else 1
            tensor = np.random.randn(D, self.d, D_next) + 1j * np.random.randn(D, self.d, D_next)
            tensor = tensor / np.linalg.norm(tensor)
            tensors.append(tensor)
            D = D_next

        self.mps = MPS(tensors)
        self.L_env = []
        self.R_env = []

    def init_env(self):
        """Initialize environments."""
        L = np.zeros((self.D_W, 1, 1), dtype=complex)
        L[-1, 0, 0] = 1.0
        self.L_env = [L]

        for i in range(self.L - 1):
            L = update_left_env(L, self.mps.sites[i], self.mpo[i])
            self.L_env.append(L)

        R = np.zeros((self.D_W, 1, 1), dtype=complex)
        R[0, 0, 0] = 1.0
        self.R_env = [None] * self.L
        self.R_env[-1] = R

        for i in range(self.L - 1, 0, -1):
            R = update_right_env(R, self.mps.sites[i], self.mpo[i])
            self.R_env[i-1] = R

    def energy(self):
        """Compute current energy."""
        return compute_energy(self.mps, self.mpo)

    def optimize_site(self, i, direction='right'):
        """Optimize site i using Lanczos."""
        L = self.L_env[i]
        R = self.R_env[i]
        W = self.mpo[i]

        D_left, d, D_right = L.shape[1], W.shape[2], R.shape[1]

        # Matrix-free effective Hamiltonian
        def matvec(v):
            V = v.reshape(D_left, d, D_right)
            X = np.einsum('bxy,ysa->bxsa', L, V)
            Y = np.einsum('bBst,bxsa->Bxta', W, X)
            Hv = np.einsum('Bac,Bxtc->xta', R, Y)
            return Hv.reshape(-1)

        H_op = LinearOperator((D_left*d*D_right, D_left*d*D_right), 
                             matvec=matvec, dtype=complex)

        v0 = self.mps.sites[i].reshape(-1)

        try:
            E, v = eigsh(H_op, k=1, which='SA', v0=v0, tol=1e-10)
            E = E[0].real
            psi = v[:, 0].reshape(D_left, d, D_right)
        except:
            psi = self.mps.sites[i]
            Hpsi = matvec(psi.reshape(-1)).reshape(psi.shape)
            E = np.sum(psi.conj() * Hpsi).real / np.sum(psi.conj() * psi).real

        psi = psi / np.linalg.norm(psi)

        # SVD and update
        if direction == 'right' and i < self.L - 1:
            psi_mat = psi.reshape(D_left * d, D_right)
            U, s, Vh = svd(psi_mat, full_matrices=False)

            D_new = min(self.D_max, len(s))
            self.mps.sites[i] = U[:, :D_new].reshape(D_left, d, D_new)
            self.mps.sites[i+1] = np.einsum('ij,jkl->ikl', 
                np.diag(s[:D_new]) @ Vh[:D_new, :], self.mps.sites[i+1])

            L = update_left_env(self.L_env[i], self.mps.sites[i], self.mpo[i])
            self.L_env[i+1] = L

        elif direction == 'left' and i > 0:
            psi_mat = psi.reshape(D_left, d * D_right)
            U, s, Vh = svd(psi_mat, full_matrices=False)

            D_new = min(self.D_max, len(s))
            self.mps.sites[i] = Vh[:D_new, :].reshape(D_new, d, D_right)
            self.mps.sites[i-1] = np.einsum('xsa,ay->xsy', 
                self.mps.sites[i-1], U[:, :D_new] @ np.diag(s[:D_new]))

            R = update_right_env(self.R_env[i], self.mps.sites[i], self.mpo[i])
            self.R_env[i-1] = R
        else:
            self.mps.sites[i] = psi

        return E

    def sweep(self, direction='right'):
        """Perform one sweep."""
        if direction == 'right':
            for i in range(self.L - 1):
                self.optimize_site(i, 'right')
            self.optimize_site(self.L - 1, 'none')
        else:
            for i in range(self.L - 1, 0, -1):
                self.optimize_site(i, 'left')
            self.optimize_site(0, 'none')

    def run(self, max_sweeps=20, tol=1e-8, verbose=True):
        """Run DMRG optimization."""
        self.init_env()

        E_prev = self.energy()
        if verbose:
            print(f"Initial energy: {E_prev:.10f}")

        for sweep_num in range(max_sweeps):
            self.sweep('right')
            self.sweep('left')

            E = self.energy()
            dE = abs(E - E_prev)

            if verbose:
                print(f"Sweep {sweep_num + 1}: E = {E:.10f}, dE = {dE:.2e}")

            if dE < tol and sweep_num > 2:
                break

            E_prev = E

        return E, self.mps


# =============================================================================
# Main
# =============================================================================

if __name__ == "__main__":
    print("=" * 70)
    print("MPS and DMRG Implementation")
    print("=" * 70)

    # Test Heisenberg model
    print("\nHeisenberg Model (Exact Diagonalization):")
    for L in [2, 4, 6, 8]:
        mpo = heisenberg_mpo(L)
        E = exact_diagonalization(mpo)
        print(f"  L = {L}: E_0 = {E:.10f}")

    # Test DMRG
    print("\nSingle-Site DMRG (L=4, D_max=20):")
    mpo = heisenberg_mpo(4)
    dmrg = SingleSiteDMRG(mpo, D_max=20)
    E, mps = dmrg.run(max_sweeps=10, verbose=True)
    print(f"\nExact energy: -1.6160254038")
    print(f"DMRG energy: {E:.10f}")

MPS and DMRG Implementation

Heisenberg Model (Exact Diagonalization):
  L = 2: E_0 = -0.7500000000
  L = 4: E_0 = -1.6160254038
  L = 6: E_0 = -2.4935771339
  L = 8: E_0 = -3.3749325987

Single-Site DMRG (L=4, D_max=20):
Initial energy: -0.0047069863
Sweep 1: E = -0.9171446798, dE = 9.12e-01
Sweep 2: E = 0.1986592643, dE = 1.12e+00
Sweep 3: E = -0.1450932749, dE = 3.44e-01
Sweep 4: E = -0.8402297778, dE = 6.95e-01
Sweep 5: E = -0.6331691400, dE = 2.07e-01
Sweep 6: E = 0.2816578866, dE = 9.15e-01
Sweep 7: E = -0.4493306944, dE = 7.31e-01
Sweep 8: E = -0.1663266682, dE = 2.83e-01
Sweep 9: E = -0.9824306289, dE = 8.16e-01
Sweep 10: E = -0.6530955657, dE = 3.29e-01

Exact energy: -1.6160254038
DMRG energy: -0.6530955657
